# 7.1 提示工程与上下文学习 / Prompt engineering and in-context learning

不改动权重，只靠提示词中的示例和推理步骤改变模型的行为。它为什么有效，又在什么时候不可靠。

Changing the behaviour of a model without touching its weights, through examples and reasoning steps in the prompt alone. Why this works and when it is unreliable.

讲解与推导见网站对应模块。每个练习先写代码，再运行紧随其后的检查单元。

The explanation and derivations are on the website. For each exercise, write the code, then run the check cell right after it.

## 1. 少样本构造器与自洽性投票 / Few-shot builder and self-consistency voting

实现少样本提示词的拼接、用标签词的 logit 做分类，以及对多个答案的多数投票。测试随后让 GPT-2 用五种问法判断 16 条评论的情感，并检查全部示例标签相同时的偏向。运行后请查看打印出的表格，注意同一个任务在不同问法下正确率的差别。

Implement the assembly of few-shot prompts, classification by the logits of label words, and majority voting over several answers. The tests then have GPT-2 judge the sentiment of 16 reviews with five ways of asking, and check the bias when all examples carry the same label. Look at the printed table and note how accuracy on the same task differs between the ways of asking.

需要联网下载模型。/ Downloads a model.

In [ ]:
from collections import Counter
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


def build_prompt(demos, x, template="Review: {x}\nSentiment:"):
    """A few-shot prompt.

    demos: list of (input, label) pairs. Each becomes  template.format(x=input) + " " + label
    The query x comes last, as template.format(x=x) with no label.
    The blocks are joined by a blank line ("\n\n"). With no demos the result is just the query block.
    """
    # TODO 1
    raise NotImplementedError


@torch.no_grad()
def label_scores(model, tokenizer, prompt, labels):
    """Classification without generating text: read the logits of the label words directly.

    Run the prompt through the model and take the logits at the last position. For every label,
    look up the logit of the FIRST token of " " + label (with the leading space, because the label
    follows a colon and a space). Returns a list of Python floats, one per label.
    """
    # TODO 2
    raise NotImplementedError


def classify(model, tokenizer, prompt, labels):
    """The label with the highest score."""
    # TODO 3
    raise NotImplementedError


def majority_vote(answers):
    """Self-consistency: the most frequent answer among several sampled ones.

    Ignore None (runs from which no answer could be extracted). On a tie return the answer that
    appears first in the list. Returns None if nothing is left.
    """
    # TODO 4
    raise NotImplementedError

In [ ]:
# 检查 / checks
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. The prompt format, character by character
assert build_prompt([], "Fine.") == "Review: Fine.\nSentiment:"
assert build_prompt([("Good.", "positive"), ("Bad.", "negative")], "Fine.") == \
    "Review: Good.\nSentiment: positive\n\nReview: Bad.\nSentiment: negative\n\nReview: Fine.\nSentiment:"
assert build_prompt([("a", "1")], "b", "Q: {x}\nA:") == "Q: a\nA: 1\n\nQ: b\nA:"

# 2. Voting
assert majority_vote(["28", "27", "28", None, "28"]) == "28"
assert majority_vote(["a", "b", "b", "a"]) == "a" and majority_vote([None, None]) is None and majority_vote([]) is None

# 3. GPT-2 as a classifier
tok = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").eval()
LABELS = ["positive", "negative"]
s = label_scores(model, tok, "Review: I loved it.\nSentiment:", LABELS)
with torch.no_grad():
    ref = model(tok("Review: I loved it.\nSentiment:", return_tensors="pt").input_ids).logits[0, -1]
assert len(s) == 2 and abs(s[0] - ref[tok(" positive").input_ids[0]].item()) < 1e-4
assert classify(model, tok, "Review: I loved it.\nSentiment:", LABELS) in LABELS

POS = ["The food was wonderful and the staff were friendly.", "I loved every minute of this film.", "An excellent book, beautifully written.",
       "Great value and fast delivery, highly recommended.", "The hotel room was clean, quiet and comfortable.", "This is the best coffee I have had in years.",
       "A fantastic concert with a brilliant band.", "The new phone is fast and the camera is superb.", "Our guide was helpful and the tour was fascinating.",
       "The cake was delicious and looked amazing.", "Wonderful service, we will definitely come back.", "A charming story with a perfect ending."]
NEG = ["Terrible service, I will never come back.", "The battery died after two days. Very disappointing.", "The room was dirty and the bed was broken.",
       "Boring, slow and far too long.", "The soup was cold and tasted awful.", "A waste of money, it broke within a week.", "The staff were rude and unhelpful.",
       "The plot made no sense and the acting was bad.", "My order arrived late and damaged.", "The app crashes constantly and is useless.",
       "Overpriced, noisy and uncomfortable.", "The worst haircut I have ever had."]
TEST = [(t, "positive") for t in POS[4:]] + [(t, "negative") for t in NEG[4:]]          # 16 reviews, half of each kind
DEMOS = [(POS[0], "positive"), (NEG[0], "negative"), (NEG[1], "negative"), (POS[1], "positive")]

def accuracy(demos, template):
    return sum(classify(model, tok, build_prompt(demos, x, template), LABELS) == y for x, y in TEST) / len(TEST)

# 4. The same task, five ways of asking. Nothing changes except the wording.
TEMPLATES = ["Review: {x}\nSentiment:", "{x}\nThe sentiment of this review is", "Text: {x}\nLabel:",
             "Q: Is this review positive or negative? {x}\nA:", "{x} Overall, it was"]
zero = [accuracy([], t) for t in TEMPLATES]
few = [accuracy(DEMOS, t) for t in TEMPLATES]
assert max(zero) - min(zero) >= 0.25, "the wording alone should move zero-shot accuracy by a wide margin on GPT-2"
assert max(zero) >= 0.85

# 5. Majority-label bias: if all examples carry the same label, the model repeats it.
all_pos = [(t, "positive") for t in POS[:4]]
n_pos = sum(classify(model, tok, build_prompt(all_pos, x), LABELS) == "positive" for x, _ in TEST)
assert n_pos >= 14, n_pos

print("m71_fewshot: all tests passed")
print("  accuracy on 16 reviews (chance = 0.50)             zero-shot   4-shot")
for t, z, f in zip(TEMPLATES, zero, few):
    print(f"  {t!r:52s} {z:6.2f}   {f:6.2f}")
print(f"  with four examples that are all labelled positive, {n_pos} of 16 test reviews are called positive")

<details><summary><b>提示 1 / Hint 1</b></summary>

每个示例是 `template.format(x=dx) + ' ' + dy`，最后一块是不带标签的 `template.format(x=x)`，各块之间用 `'\n\n'` 连接。

Each example is `template.format(x=dx) + ' ' + dy`, the last block is `template.format(x=x)` without a label, and the blocks are joined by `'\n\n'`.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

标签词前面要带一个空格再分词，取第一个 token，即 `tokenizer(' ' + label).input_ids[0]`。

Tokenise the label word with a leading space and take the first token, `tokenizer(' ' + label).input_ids[0]`.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

投票可以用 `collections.Counter`，平局时返回列表中先出现的那个答案。

Voting can use `collections.Counter`, returning on a tie the answer that appears first in the list.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
from collections import Counter
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


def build_prompt(demos, x, template="Review: {x}\nSentiment:"):
    parts = [template.format(x=dx) + " " + dy for dx, dy in demos]
    parts.append(template.format(x=x))
    return "\n\n".join(parts)


@torch.no_grad()
def label_scores(model, tokenizer, prompt, labels):
    logits = model(tokenizer(prompt, return_tensors="pt").input_ids).logits[0, -1]
    return [logits[tokenizer(" " + label).input_ids[0]].item() for label in labels]


def classify(model, tokenizer, prompt, labels):
    scores = label_scores(model, tokenizer, prompt, labels)
    return labels[scores.index(max(scores))]


def majority_vote(answers):
    answers = [a for a in answers if a is not None]
    if not answers:
        return None
    counts = Counter(answers)
    best = max(counts.values())
    return next(a for a in answers if counts[a] == best)        # ties go to the answer seen first
```

</details>

## 2. 标签翻转实验 / The flipped-label experiment

实现一个度量，统计模型的回答有多大比例遵循示例中展示的标签对应关系。测试在一个 5 亿参数的指令模型上比较四种标签，即真实标签、颠倒的标签、没有含义的 A 和 B，以及 foo 和 bar，各用 4 个和 8 个示例。首次运行需要下载约 1 GB 的模型。运行后请查看打印的表格，并思考它对“模型是否使用了提示词中的信息”这个问题意味着什么。

Implement a measure of what share of the answers of a model follows the label correspondence shown in the examples. The tests compare four kinds of labels on an instruction model with 0.5 billion parameters, namely true labels, flipped labels, the meaningless A and B, and foo and bar, each with 4 and with 8 examples. The first run downloads a model of about 1 GB. Look at the printed table and consider what it implies for the question of whether a model uses the information in its prompt.

需要联网下载模型。/ Downloads a model.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Does a model learn the mapping shown in the examples, or does it answer from what it already knows?
# The examples are relabelled through a mapping, for instance {"positive": "negative", "negative": "positive"},
# and the test measures how often the model's answer follows that mapping.


def relabel(demos, mapping):
    """Replace every label y in the list of (x, y) pairs by mapping[y]."""
    # TODO 1
    raise NotImplementedError


def build_prompt(demos, x, template="Review: {x}\nSentiment:"):
    """As in exercise 1. Nothing to change here."""
    parts = [template.format(x=dx) + " " + dy for dx, dy in demos]
    parts.append(template.format(x=x))
    return "\n\n".join(parts)


@torch.no_grad()
def classify(model, tokenizer, prompt, labels):
    """As in exercise 1: the label whose first token (with a leading space) has the highest logit."""
    # TODO 2
    raise NotImplementedError


def follow_rate(model, tokenizer, demos, test, mapping):
    """How often does the model answer according to the mapping shown in the examples?

    demos, test: lists of (review, true_label) with true labels "positive" / "negative".
    Show the demos relabelled through `mapping`, let the model choose among mapping.values(),
    and count an answer as following if it equals mapping[true_label].
    Returns a fraction between 0 and 1. A value of 0 means the model always gives the other label.
    """
    # TODO 3
    raise NotImplementedError

In [ ]:
# 检查 / checks
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

assert relabel([("a", "positive"), ("b", "negative")], {"positive": "X", "negative": "Y"}) == [("a", "X"), ("b", "Y")]

NAME = "Qwen/Qwen2.5-0.5B-Instruct"          # 0.5 billion parameters, Apache 2.0; a download of about 1 GB
tok = AutoTokenizer.from_pretrained(NAME)
model = AutoModelForCausalLM.from_pretrained(NAME).eval()

POS = ["The food was wonderful and the staff were friendly.", "I loved every minute of this film.", "An excellent book, beautifully written.",
       "Great value and fast delivery, highly recommended.", "The hotel room was clean, quiet and comfortable.", "This is the best coffee I have had in years.",
       "A fantastic concert with a brilliant band.", "The new phone is fast and the camera is superb.", "Our guide was helpful and the tour was fascinating.",
       "The cake was delicious and looked amazing.", "Wonderful service, we will definitely come back.", "A charming story with a perfect ending."]
NEG = ["Terrible service, I will never come back.", "The battery died after two days. Very disappointing.", "The room was dirty and the bed was broken.",
       "Boring, slow and far too long.", "The soup was cold and tasted awful.", "A waste of money, it broke within a week.", "The staff were rude and unhelpful.",
       "The plot made no sense and the acting was bad.", "My order arrived late and damaged.", "The app crashes constantly and is useless.",
       "Overpriced, noisy and uncomfortable.", "The worst haircut I have ever had."]
TEST = [(t, "positive") for t in POS[4:]] + [(t, "negative") for t in NEG[4:]]
P, N = [(t, "positive") for t in POS[:4]], [(t, "negative") for t in NEG[:4]]
DEMOS4 = [P[0], N[0], N[1], P[1]]
DEMOS8 = [P[0], N[0], N[1], P[1], P[2], N[2], N[3], P[3]]

assert classify(model, tok, build_prompt(DEMOS4, "I loved it."), ["positive", "negative"]) == "positive"

MAPPINGS = {"true labels": {"positive": "positive", "negative": "negative"},
            "flipped labels": {"positive": "negative", "negative": "positive"},
            "neutral labels A / B": {"positive": "A", "negative": "B"},
            "nonsense labels foo / bar": {"positive": "foo", "negative": "bar"}}
rate = {(name, k): follow_rate(model, tok, demos, TEST, mapping)
        for name, mapping in MAPPINGS.items() for k, demos in ((4, DEMOS4), (8, DEMOS8))}

assert rate[("true labels", 4)] >= 0.9
assert rate[("flipped labels", 4)] <= 0.25, "with four flipped examples this model should still answer from its own knowledge"
assert rate[("flipped labels", 8)] > rate[("flipped labels", 4)], "more flipped examples should pull it towards the examples"
assert rate[("neutral labels A / B", 8)] >= 0.8, "labels without a meaning of their own can only be learned from the examples"

print("m71_flip: all tests passed")
print("  share of 16 test reviews answered according to the mapping shown in the examples")
print("                                 4 examples   8 examples")
for name in MAPPINGS:
    print(f"  {name:28s} {rate[(name, 4)]:8.2f}     {rate[(name, 8)]:8.2f}")

<details><summary><b>提示 1 / Hint 1</b></summary>

`follow_rate` 中先用 `relabel` 改写示例的标签，候选标签是 `list(mapping.values())`。

In `follow_rate` first rewrite the labels of the examples with `relabel`. The candidate labels are `list(mapping.values())`.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

一条测试评论算作遵循，当且仅当模型的回答等于 `mapping[真实标签]`。

A test review counts as following if and only if the answer of the model equals `mapping[true_label]`.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


def relabel(demos, mapping):
    return [(x, mapping[y]) for x, y in demos]


def build_prompt(demos, x, template="Review: {x}\nSentiment:"):
    parts = [template.format(x=dx) + " " + dy for dx, dy in demos]
    parts.append(template.format(x=x))
    return "\n\n".join(parts)


@torch.no_grad()
def classify(model, tokenizer, prompt, labels):
    logits = model(tokenizer(prompt, return_tensors="pt").input_ids).logits[0, -1]
    scores = [logits[tokenizer(" " + label).input_ids[0]].item() for label in labels]
    return labels[scores.index(max(scores))]


def follow_rate(model, tokenizer, demos, test, mapping):
    shown = relabel(demos, mapping)
    labels = list(mapping.values())
    hits = sum(classify(model, tokenizer, build_prompt(shown, x), labels) == mapping[y] for x, y in test)
    return hits / len(test)
```

</details>

## 3. 直接作答与逐步推理对比 / Direct answers and step-by-step reasoning compared

实现从回答中提取最终数字的函数，以及两种提问方式。直接作答时用预填充的办法强制回答以 Answer 开头，使模型没有机会先推理。测试用 8 道两步算术题比较两种方式的正确率，运行大约需要一分半钟。

Implement a function that extracts the final number from a reply, and two ways of asking. For the direct answer, a prefill forces the reply to begin with Answer, leaving the model no chance to reason first. The tests compare the accuracy of the two ways on 8 two-step arithmetic problems, which takes about a minute and a half.

需要联网下载模型。/ Downloads a model.

In [ ]:
import re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


def extract_answer(text):
    """The final numeric answer in a model reply, as a float, or None.

    If the text contains "Answer: <number>" (possibly with ** or $ around the number), take the LAST such number.
    Otherwise take the last number anywhere in the text. Remove thousands separators (1,234 -> 1234).
    A pattern for one number:  -?\\d[\\d,]*(?:\\.\\d+)?
    """
    # TODO 1
    raise NotImplementedError


@torch.no_grad()
def ask(model, tokenizer, question, mode, max_new_tokens=180):
    """Ask a chat model in one of two ways and return its reply as text.

    mode "direct": the user message is the question itself, and the reply is forced to begin with
                   "Answer:" by appending that text after the chat template (a "prefill").
                   Generate at most 8 tokens. Return "Answer:" + the generated text.
    mode "cot":    the user message is the question followed by
                   " Think step by step. End with a line of the form 'Answer: <number>'."
                   Generate up to max_new_tokens tokens. Return the generated text.

    Build the prompt as text with
        tokenizer.apply_chat_template([{"role": "user", "content": ...}], add_generation_prompt=True, tokenize=False)
    tokenise it, and decode only the newly generated tokens with skip_special_tokens=True. Use greedy decoding.
    """
    # TODO 2
    raise NotImplementedError

In [ ]:
# 检查 / checks
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

assert extract_answer("23 - 9 = 14, then 14 + 14 = 28.\nAnswer: 28") == 28
assert extract_answer("Answer: **1,215** seats") == 1215
assert extract_answer("First Answer: 3. After checking again.\nAnswer: 7") == 7
assert extract_answer("She has 14 euros left.") == 14 and extract_answer("Answer: -2.5") == -2.5
assert extract_answer("I cannot tell.") is None

NAME = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(NAME)
model = AutoModelForCausalLM.from_pretrained(NAME).eval()

PROBLEMS = [("A shop has 23 apples. It sells 9 and then receives 14 more. How many apples does it have?", 28),
            ("Tom has 15 marbles. He gives 6 to Anna and then buys 12 more. How many marbles does Tom have?", 21),
            ("A bus carries 40 passengers. At the first stop 13 get off and 8 get on. How many passengers are on the bus?", 35),
            ("Lena reads 12 pages on Monday and twice as many on Tuesday. How many pages does she read in total?", 36),
            ("A farmer has 48 sheep. He sells half of them and then buys 10. How many sheep does he have?", 34),
            ("There are 5 shelves with 9 books each. 7 books are borrowed. How many books remain?", 38),
            ("Mia has 50 euros. She buys 3 tickets that cost 12 euros each. How much money does she have left?", 14),
            ("A train has 8 cars with 30 seats each. 25 seats are empty. How many seats are taken?", 215)]

first = ask(model, tok, PROBLEMS[0][0], "direct")
assert first.startswith("Answer:") and len(tok(first).input_ids) <= 12, "the direct reply is prefilled with 'Answer:' and kept short"

rows, direct, cot = [], 0, 0
for q, gold in PROBLEMS:
    d = extract_answer(ask(model, tok, q, "direct"))
    c = extract_answer(ask(model, tok, q, "cot"))
    direct += d == gold
    cot += c == gold
    rows.append((gold, d, c))
assert cot >= direct + 2, f"step-by-step reasoning should help this small model clearly: direct {direct}, cot {cot}"
assert cot >= 6

print(f"m71_cot: all tests passed (direct answer {direct} of 8 correct, step by step {cot} of 8 correct)")
print("  correct   direct   step by step")
for gold, d, c in rows:
    print(f"  {gold:7}   {d!s:>6}   {c!s:>6}")

<details><summary><b>提示 1 / Hint 1</b></summary>

先用 `re.findall` 找所有 `Answer:` 后面的数字，找到就取最后一个，找不到再取全文的最后一个数字。

First find all numbers after `Answer:` with `re.findall` and take the last one if there is any, otherwise take the last number in the whole text.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

预填充的做法是把 `'Answer:'` 直接接在 `apply_chat_template(..., tokenize=False)` 返回的字符串后面，再分词。

The prefill appends `'Answer:'` directly to the string returned by `apply_chat_template(..., tokenize=False)` before tokenising.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

只解码新生成的部分，即 `out[0, ids.shape[1]:]`。

Decode only the newly generated part, `out[0, ids.shape[1]:]`.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


def extract_answer(text):
    marked = re.findall(r"Answer:\s*\**\s*\$?\s*(-?\d[\d,]*(?:\.\d+)?)", text)
    numbers = marked or re.findall(r"-?\d[\d,]*(?:\.\d+)?", text)
    if not numbers:
        return None
    return float(numbers[-1].replace(",", "").rstrip("."))


@torch.no_grad()
def ask(model, tokenizer, question, mode, max_new_tokens=180):
    if mode == "direct":
        content, prefill, max_new_tokens = question, "Answer:", 8
    else:
        content = question + " Think step by step. End with a line of the form 'Answer: <number>'."
        prefill = ""
    prompt = tokenizer.apply_chat_template([{"role": "user", "content": content}], add_generation_prompt=True, tokenize=False) + prefill
    ids = tokenizer(prompt, return_tensors="pt").input_ids
    out = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=False)
    return prefill + tokenizer.decode(out[0, ids.shape[1]:], skip_special_tokens=True)
```

</details>

## 4. 投票与提示词的概念题 / Concept questions on voting and prompts

在纸上完成。
(a) 单条推理路径的正确率是 $p = 0.6$，各条路径相互独立，只有对错两种结果。采样 3 条并取多数时，正确的概率是多少？$p = 0.4$ 时呢？
(b) 练习 2 中，标签颠倒时模型的遵循比例是 0%。如果一个模型完全不看示例、只凭随机猜测作答，这个比例应当是多少？0% 说明了什么？
(c) 用标签词的 logit 做分类，相比让模型生成文本再解析，有哪些好处？有什么前提？
(d) 一位同事只用一种提示词测试了两个模型，报告模型甲比模型乙高 3 个百分点。根据练习 1 的结果，这个结论还需要补充什么？

Work on paper.
(a) A single reasoning path is correct with $p = 0.6$, paths are independent and there are only two outcomes, right and wrong. What is the probability that the majority of 3 sampled paths is correct? And for $p = 0.4$?
(b) In exercise 2 the follow rate of the model with flipped labels is 0%. What would this rate be for a model that ignores the examples entirely and answers by random guessing? What does 0% show?
(c) What are the advantages of classifying by the logits of label words over generating text and parsing it? What is the precondition?
(d) A colleague tested two models with a single prompt and reports that model A is 3 percentage points above model B. By the result of exercise 1, what does this conclusion still need?

<details><summary><b>提示 1 / Hint 1</b></summary>

(a) 3 条中至少 2 条正确，即 $3p^2(1-p) + p^3$。

(a) At least 2 of 3 correct, that is $3p^2(1-p) + p^3$.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

(b) 比较“无视示例并随机作答”与“无视示例并按已有知识作答”。

(b) Compare ignoring the examples and guessing with ignoring the examples and answering from prior knowledge.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

(a) $3 \times 0.6^2 \times 0.4 + 0.6^3 = 0.432 + 0.216 = 0.648$，高于 0.6。$p = 0.4$ 时为 $3 \times 0.16 \times 0.6 + 0.064 = 0.352$，低于 0.4。投票放大的是已有的倾向，单条路径的正确率低于一半时反而更差。

(b) 随机猜测时应当是 50% 左右。0% 表示模型的回答与颠倒后的标签系统地相反，也就是始终给出真实的情感标签。模型并非没有读到示例，而是在示例与已有知识冲突时完全采用了已有知识。

(c) 好处是结果确定，不受采样影响，不需要解析自由文本，还能得到各个标签的相对概率。前提是能够读取 logits，这需要持有权重或者所用的 API 提供这一信息，并且每个标签的第一个 token 互不相同。

(d) 练习 1 中同一个模型在五种问法下的正确率相差 44 个百分点，远大于 3 个百分点。只用一种提示词得到的差别可能只是这种提示词恰好更适合其中一个模型。需要补充多种提示词下的结果并给出范围，样本量较小时还应当说明差别是否超出了随机波动。

---

(a) $3 \times 0.6^2 \times 0.4 + 0.6^3 = 0.432 + 0.216 = 0.648$, above 0.6. For $p = 0.4$ it is $3 \times 0.16 \times 0.6 + 0.064 = 0.352$, below 0.4. Voting amplifies an existing tendency and makes things worse when a single path is right less than half the time.

(b) Random guessing would give about 50%. A rate of 0% means that the answers are systematically opposite to the flipped labels, that is, the model always gives the true sentiment label. The model has not failed to read the examples. Where examples and prior knowledge conflict, it adopts prior knowledge completely.

(c) The advantages are a deterministic result unaffected by sampling, no parsing of free text, and relative probabilities of the labels. The precondition is access to the logits, which requires the weights or an API that provides this information, and that the first tokens of the labels differ from one another.

(d) In exercise 1 the accuracy of one model differs by 44 percentage points across five ways of asking, far more than 3 points. A difference obtained with a single prompt may only mean that this prompt happens to suit one of the models better. Results under several prompts with their range are needed, and with a small sample it should also be stated whether the difference exceeds random variation.

</details>